# Notebook 4 - Levantamento de hipóteses

Este notebook reúne a verificação, sobre os dados, das três hipóteses levantadas pelo grupo a respeito da relação entre as variáveis do conjunto e o problema de validação térmica da Whirlpool. O texto de cada hipótese acompanha o código que a sustenta, e as conclusões são retomadas na Seção 4.2.3 da documentação.

As hipóteses foram escolhidas por serem sustentadas por evidências levantadas nas etapas de compreensão e preparação dos dados, e por se conectarem aos objetivos definidos com a parceira no kickoff do projeto.

| Hipótese | Enunciado |
|---|---|
| 1 | A área do produto, Key Area, é o fator com maior poder explicativo sobre a temperatura |
| 2 | O tipo de teste, Run Case, determina o patamar da temperatura |
| 3 | A distância até o limite de segurança é mais relevante para o problema do que a temperatura absoluta |

## Setup do ambiente

A base utilizada é a exportada pelo notebook de leitura e limpeza, de modo que a verificação parta exatamente do mesmo conjunto de leituras válidas empregado nas demais análises.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

DIR_SAIDA = "dados_intermediarios"
CAMINHO_LIMPO = os.path.join(DIR_SAIDA, "df_limpo.csv")

if not os.path.exists(CAMINHO_LIMPO):
    raise FileNotFoundError(
        "df_limpo.csv não encontrado. Execute o notebook leitura.ipynb antes deste."
    )

df_limpo = pd.read_csv(CAMINHO_LIMPO)
df_limpo["Possui Active Limit"] = df_limpo["Possui Active Limit"].astype(bool)

print("Base carregada:", df_limpo.shape)
print("Termopares:", df_limpo["TC Name"].nunique(), "| áreas:", df_limpo["Key Area"].nunique())

## Hipótese 1 - A área do produto é o fator com maior poder explicativo sobre a temperatura

Cada termopar está associado a uma área do produto, registrada na coluna Key Area. A hipótese é que essa área já explica boa parte da temperatura registrada, de modo que saber em que parte do fogão um sensor está posicionado permita antecipar se a leitura tende a ser alta ou baixa, antes mesmo de qualquer outra variável ser observada.

A verificação é feita em duas etapas. Primeiro, são comparadas média, mediana e dispersão entre as áreas. Em seguida, é calculada a proporção da variância total da temperatura que é explicada pela área, por meio da razão entre a soma dos quadrados entre grupos e a soma dos quadrados total.

In [ ]:
estatisticas_area = (
    df_limpo.groupby("Key Area")["Temperature"]
    .agg(n="count", media="mean", mediana="median", desvio="std", minimo="min", maximo="max")
    .round(2)
    .sort_values("media", ascending=False)
)
print("ESTATÍSTICAS DA TEMPERATURA POR ÁREA")
display(estatisticas_area)

amplitude_medias = estatisticas_area["media"].max() - estatisticas_area["media"].min()
print(f"Amplitude entre as médias das áreas: {amplitude_medias:.2f} °C")
print(f"Menor média: {estatisticas_area['media'].min():.2f} °C em {estatisticas_area['media'].idxmin()}")
print(f"Maior média: {estatisticas_area['media'].max():.2f} °C em {estatisticas_area['media'].idxmax()}")

In [ ]:
# Proporção da variância da temperatura explicada pela área (eta ao quadrado)
media_geral = df_limpo["Temperature"].mean()

sq_total = ((df_limpo["Temperature"] - media_geral) ** 2).sum()
sq_entre = sum(
    len(grupo) * (grupo.mean() - media_geral) ** 2
    for _, grupo in df_limpo.groupby("Key Area")["Temperature"]
)
eta2_area = sq_entre / sq_total

print(f"Variância da temperatura explicada pela Key Area: {eta2_area:.1%}")

In [ ]:
df_limpo.boxplot(column="Temperature", by="Key Area", figsize=(12, 6), rot=45)
plt.title("Distribuição da temperatura por área do produto")
plt.suptitle("")
plt.xlabel("Key Area")
plt.ylabel("Temperatura (°C)")
plt.tight_layout()
plt.show()

### Conclusão da hipótese 1

A hipótese é sustentada pelos dados. As médias variam de forma acentuada entre as áreas, e a dispersão também difere: há áreas concentradas em faixa estreita, como Oven HMI e CT Knobs, e áreas muito mais espalhadas, como Coil Temps e Mainback. Cada área possui, portanto, não apenas um patamar térmico próprio, mas também um comportamento de variação próprio.

A proporção da variância explicada confirma o peso da variável, e o resultado é coerente com a arquitetura do produto descrita pela parceira no kickoff, que aponta a cavidade do forno, o módulo de indução e o console como subsistemas de risco térmico distintos.

Um reforço independente vem do teste de normalidade apresentado no notebook de normalização, cuja recomendação foi tratar a temperatura de forma segmentada por área, já que cada região possui escala e formato de distribuição próprios. Uma análise conduzida com outro objetivo chegou à mesma conclusão desta hipótese.

Para o projeto, isso indica que a Key Area deve ter peso alto na modelagem, podendo justificar inclusive modelos separados por área, o que facilitaria também a leitura dos resultados pelos times que já pensam o produto dividido nesses subsistemas.

## Hipótese 2 - O tipo de teste determina o patamar da temperatura

A coluna Run Case indica o tipo de ensaio realizado em cada simulação, sendo os principais o modo Pyro, de autolimpeza, e as variações do modo Broil, mais próximas do uso comum. A hipótese é que o tipo de ensaio altera o patamar de temperatura esperado para um mesmo sensor, de modo que não baste saber qual componente está sendo medido.

A verificação compara as distribuições entre os tipos de ensaio e, em seguida, isola o efeito dentro de um mesmo termopar, comparando as leituras de cada sensor entre os diferentes ensaios. Essa segunda etapa é a que separa o efeito do ensaio do efeito do componente.

In [ ]:
estatisticas_ensaio = (
    df_limpo.groupby("Run Case")["Temperature"]
    .agg(n="count", media="mean", mediana="median", desvio="std")
    .round(2)
    .sort_values("mediana", ascending=False)
)
print("ESTATÍSTICAS DA TEMPERATURA POR TIPO DE ENSAIO")
display(estatisticas_ensaio)

print(f"Maior mediana: {estatisticas_ensaio['mediana'].max():.2f} °C em {estatisticas_ensaio['mediana'].idxmax()}")
print(f"Menor mediana: {estatisticas_ensaio['mediana'].min():.2f} °C em {estatisticas_ensaio['mediana'].idxmin()}")
print(f"Maior dispersão: {estatisticas_ensaio['desvio'].max():.2f} °C em {estatisticas_ensaio['desvio'].idxmax()}")

In [ ]:
# Efeito do ensaio dentro de um mesmo termopar: para cada sensor, é medida a
# diferença entre a maior e a menor média por tipo de ensaio.
medias_sensor_ensaio = df_limpo.groupby(["TC Name", "Run Case"])["Temperature"].mean()
variacao_interna = (
    medias_sensor_ensaio.groupby("TC Name").agg(lambda s: s.max() - s.min())
    .sort_values(ascending=False)
    .round(2)
)

print("Variação da média de um mesmo termopar entre tipos de ensaio")
print(f"Mediana da variação: {variacao_interna.median():.2f} °C")
print(f"Termopares com variação acima de 20 °C: {(variacao_interna > 20).sum()} de {len(variacao_interna)}")
print()
print("Dez termopares mais sensíveis ao tipo de ensaio:")
display(variacao_interna.head(10).to_frame("Variação entre ensaios (°C)"))

In [ ]:
df_limpo.boxplot(column="Temperature", by="Run Case", figsize=(10, 6), rot=45)
plt.title("Distribuição da temperatura por tipo de ensaio")
plt.suptitle("")
plt.xlabel("Run Case")
plt.ylabel("Temperatura (°C)")
plt.tight_layout()
plt.show()

### Conclusão da hipótese 2

A hipótese é sustentada. As medianas e as dispersões diferem entre os tipos de ensaio, e a comparação interna a cada termopar mostra que a diferença não decorre apenas de os ensaios cobrirem conjuntos distintos de sensores: um mesmo componente registra médias bastante diferentes conforme o ensaio.

Duas evidências obtidas em outras etapas convergem para a mesma conclusão. A primeira é o teste de normalidade, em que a temperatura reprova o Shapiro-Wilk em todas as áreas, comportamento compatível com distribuições multimodais formadas pela mistura de ensaios com patamares térmicos distintos. A segunda é o tratamento de outliers, no qual o agrupamento apenas por termopar produzia falsos positivos, corrigidos somente quando o agrupamento passou a considerar também o tipo de ensaio.

Para o projeto, isso significa que o modelo não pode tratar todas as leituras de um sensor de forma unificada. O patamar de referência é ditado pelo tipo de ensaio, e os desvios em relação a ele decorrem das variações físicas da configuração. Ignorar essa variável levaria o modelo a errar previsões para os usos cotidianos, comprometendo a meta de margem de erro de até 10%.

## Hipótese 3 - A distância até o limite de segurança é mais relevante do que a temperatura absoluta

O Active Limit é a temperatura máxima que um componente pode atingir antes de correr risco de falha, definida por norma técnica ou pela especificação do fornecedor. A hipótese é que, para decidir se um design é seguro, não basta conhecer a temperatura medida: o que importa é a distância entre essa temperatura e o limite daquele componente específico.

A verificação busca casos concretos de leituras com temperatura próxima entre si e situação de risco oposta, o que só é possível se o limite variar entre componentes. Em seguida, é quantificada a parcela de leituras que ultrapassa o limite e a que opera dentro da margem de segurança de 10% exigida pela engenharia.

In [ ]:
com_limite = df_limpo[df_limpo["Possui Active Limit"]].copy()

print("Leituras com limite de segurança aplicável:", len(com_limite), "de", len(df_limpo))
print("Limites distintos na base:", com_limite["Active Limit"].nunique())
print("Faixa dos limites:", com_limite["Active Limit"].min(), "°C a", com_limite["Active Limit"].max(), "°C")
print()

# Duas leituras de temperatura próxima e situação de risco oposta
ultrapassa = com_limite[com_limite["Margem de Seguranca"] < 0]
folgada = com_limite[com_limite["Margem de Seguranca"] > 40]

alvo = ultrapassa["Temperature"].median()
caso_risco = ultrapassa.iloc[(ultrapassa["Temperature"] - alvo).abs().argsort()[:1]]
caso_folga = folgada.iloc[(folgada["Temperature"] - alvo).abs().argsort()[:1]]

colunas = ["TC Name", "Key Area", "Temperature", "Active Limit", "Margem de Seguranca"]
print("Duas leituras em temperatura semelhante e situação de risco oposta:")
display(pd.concat([caso_risco, caso_folga])[colunas].round(2))

In [ ]:
# Classificação das leituras conforme a distância até o limite
dentro_margem = com_limite["Temperature"] <= 0.9 * com_limite["Active Limit"]
entre_margem_e_limite = (com_limite["Temperature"] > 0.9 * com_limite["Active Limit"]) & (com_limite["Margem de Seguranca"] >= 0)
acima_limite = com_limite["Margem de Seguranca"] < 0

resumo = pd.DataFrame({
    "Situação": ["Dentro da margem de 10%", "Entre a margem e o limite", "Acima do limite"],
    "Leituras": [int(dentro_margem.sum()), int(entre_margem_e_limite.sum()), int(acima_limite.sum())],
})
resumo["Percentual"] = (resumo["Leituras"] / len(com_limite) * 100).round(1)
display(resumo)

# Correlação entre temperatura absoluta e a margem, para mostrar que uma não substitui a outra
correlacao = com_limite["Temperature"].corr(com_limite["Margem de Seguranca"])
print(f"Correlação entre temperatura absoluta e margem de segurança: {correlacao:.3f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(com_limite["Active Limit"], com_limite["Temperature"], alpha=0.5, color="#4C72B0")

maximo = com_limite["Active Limit"].max()
plt.plot([0, maximo], [0, maximo], color="red", linestyle="--", label="Falha absoluta")
plt.plot([0, maximo], [0, maximo * 0.9], color="orange", linewidth=2, label="Margem de segurança de 10%")

plt.title("Temperatura medida em relação ao limite de segurança do componente")
plt.xlabel("Active Limit (°C)")
plt.ylabel("Temperatura medida (°C)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

### Conclusão da hipótese 3

A hipótese é sustentada. Como os limites variam entre componentes, existem leituras com temperatura semelhante em situações de risco opostas, o que mostra que o valor absoluto, isoladamente, não permite decidir sobre a segurança do design. A correlação entre a temperatura e a margem é negativa, mas está longe de ser perfeita, o que confirma que uma variável não substitui a outra.

A distribuição das leituras entre as três situações mostra ainda que existe um grupo relevante de casos que aprovam no limite absoluto mas operam fora da margem de 10% recomendada, exatamente a faixa que interessa antecipar na triagem de novos designs.

Para o projeto, essa hipótese altera a forma de medir o sucesso do modelo. O indicador definido no charter trata da margem de erro sobre a temperatura prevista, métrica que, isoladamente, pode aprovar um modelo que erra a decisão mais importante, que é a de classificar o componente como dentro ou fora da margem de segurança. A recomendação é complementar o erro percentual com uma taxa de acerto sobre aprovação e reprovação, mais alinhada ao uso que a engenharia fará do modelo.

Cabe registrar que a margem de segurança permanece restrita à análise, sem integrar o conjunto de features, uma vez que é calculada a partir da própria temperatura e configuraria vazamento de dados, conforme detalhado na Seção 4.2.2.3.

## Síntese

As três hipóteses são sustentadas pelos dados. A área do produto e o tipo de ensaio explicam, em conjunto, boa parte do patamar térmico esperado para cada leitura, e devem ter peso alto na modelagem. A distância até o limite de segurança, por sua vez, não altera as features do modelo, mas altera o critério de avaliação, ao mostrar que acertar a temperatura não equivale a acertar a decisão de segurança.

As conclusões aqui obtidas são retomadas, com o detalhamento das justificativas, na Seção 4.2.3 da documentação.